# Ajustamento pelo Método Correlato (Método das Condições)
Este notebook apresenta a resolução de uma rede de nivelamento utilizando o **Método Correlato**. 

### **O Conceito**
No método correlato, ao contrário do paramétrico, buscamos **correções (v)** para as observações que as tornem geometricamente consistentes em circuitos fechados. 
As equações de condição seguem a forma:
$$B \cdot V + W = 0$$
Onde:
- $B$: Matriz de coeficientes das condições.
- $V$: Vetor de resíduos (correções).
- $W$: Vetor de fechamento ($B \cdot L_b$).

## 1. Dados da Rede de Nivelamento

![rede de nivelamento](figura_1.png)

Conforme a imagem acima, temos 6 observações de desnível ($L_1$ a $L_6$) e seus respectivos comprimentos para cálculo dos pesos.

| Linha | De | Para | ΔH (m) | Dist (km) |
|:---:|:---:|:---:|:---:|:---:|
| L1 | A | I | 6.67 | 4 |
| L2 | A | II | 12.78 | 2 |
| L3 | I | II | 6.15 | 2 |
| L4 | A | III | 1.02 | 4 |
| L5 | III | II | 11.88 | 2 |
| L6 | III | I | 5.77 | 4 |

In [4]:
import numpy as np

# 1. Observações (Lb)
Lb = np.array([[6.67], [12.78], [6.15], [1.02], [11.88], [5.77]])

# 2. Matriz de Pesos (P)
# O peso é inversamente proporcional ao comprimento (P = 1/s)
s = np.array([4, 2, 2, 4, 2, 4])
P = np.diag(1/s)

# 3. Matriz B (Equações de Condição)
# Definimos 3 circuitos independentes baseados na rede:
# C1 (A-I-II-A): L1 + L3 - L2 = 0
# C2 (A-III-I-A): L4 - L6 - L1 = 0
# C3 (A-III-II-A): L4 + L5 - L2 = 0

B = np.array([
    [ 1, -1,  1,  0,  0,  0],
    [-1,  0,  0,  1,  0, -1],
    [ 0, -1,  0,  1,  1,  0]
])

# 4. Vetor de Fechamento (W = B * Lb)
W = B @ Lb
print("Vetor de Fechamento W (m):\n", W)

Vetor de Fechamento W (m):
 [[  0.04]
 [-11.42]
 [  0.12]]


## 2. Resolução do Sistema pelo Método dos Correlatos
A solução para o vetor de correlatos ($K$) é:
$$K = -(B \cdot P^{-1} \cdot B^T)^{-1} \cdot W$$

Em seguida, as correções ($V$) são obtidas por:
$$V = P^{-1} \cdot B^T \cdot K$$

In [2]:
# Matriz de covariância (M = P^-1)
M = np.linalg.inv(P)

# Matriz do sistema normal correlato (Mc = B * M * B.T)
Mc = B @ M @ B.T

# Calculando Correlatos (K)
K = -np.linalg.inv(Mc) @ W

# Calculando Resíduos/Correções (V)
V = M @ B.T @ K

# Observações Ajustadas (La = Lb + V)
La = Lb + V

print("Correções V (m):\n", V)
print("\nDesníveis Ajustados La (m):\n", La)

Correções V (m):
 [[-2.3  ]
 [ 0.032]
 [ 2.292]
 [ 2.236]
 [-2.324]
 [-6.884]]

Desníveis Ajustados La (m):
 [[ 4.37 ]
 [12.812]
 [ 8.442]
 [ 3.256]
 [ 9.556]
 [-1.114]]


## 3. Verificação de Resultados
Um ajuste bem-sucedido deve resultar em fechamentos nulos nos circuitos ($B \cdot L_a = 0$).

In [3]:
check = B @ La
print("Verificação de Fechamento (deve ser zero):\n", np.round(check, 10))

# Cálculo das altitudes finais (Fixando A = 100.000 m)
HA = 100.000
HI = HA + La[0,0]
HII = HA + La[1,0]
HIII = HA + La[3,0]

print(f"\nAltitudes Finais Ajustadas:")
print(f"Ponto I:   {HI:.3f} m")
print(f"Ponto II:  {HII:.3f} m")
print(f"Ponto III: {HIII:.3f} m")

Verificação de Fechamento (deve ser zero):
 [[ 0.]
 [-0.]
 [ 0.]]

Altitudes Finais Ajustadas:
Ponto I:   104.370 m
Ponto II:  112.812 m
Ponto III: 103.256 m
